# 02 · IxIDN+ / IxIDN- and IxIDN-Matched

Walk-through of the shipped benchmark tables from the clinical-trial co-investigation graph to the 4,364-pair IxIDN-Matched benchmark (Methods *IxIDN-Matched benchmark*; Supplementary *How IxIDN-Matched is built*). This notebook inspects the staged construction outputs (`data/02S_*`, `data/00_raw/frozen/`, `data/06S_final_benchmark/`); the scripted rebuild is in `code/01_build_ixidn/`.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "00_raw").is_dir())
D = REPO / "data"
RAW = D / "00_raw"
def pairs(a, b):
    return {tuple(sorted(x)) for x in zip(a, b)}
import networkx as nx

## IxIDN+ : co-investigation graph
Two diseases are linked when the same drug was tested in trials for both. Supplementary: *950,131 possible disease pairs* in this graph.

In [2]:
pos = pd.read_parquet(D / "02S_build_IxIDN/ixidn_pos_edges.parquet.gz")
G = nx.Graph(); G.add_edges_from(zip(pos.disease_1, pos.disease_2))
n = G.number_of_nodes()
print("IxIDN+ edges", G.number_of_edges(), "| diseases", n, "| possible pairs", n * (n - 1) // 2)
print("shared drugs per edge:", pos.n_drugs.describe()[["min", "50%", "max"]].to_dict())
print("max phase:", pos.max_phase_int.value_counts().sort_index().to_dict())
assert n * (n - 1) // 2 == 950131

IxIDN+ edges 43112 | diseases 1379 | possible pairs 950131
shared drugs per edge: {'min': 1.0, '50%': 2.0, 'max': 2183.0}
max phase: {1.0: 440, 2.0: 9933, 3.0: 10187, 4.0: 22538}


## IxIDN- : distant pairs, then the co-investigation veto
Candidate negatives are disease pairs at least three steps apart in the co-investigation graph (`d_sp`); the veto step then removes further pairs as described in Methods. The evaluation pair set is IxIDN+ plus the post-veto IxIDN-.

In [3]:
pin = pd.read_parquet(D / "02S_build_IxIDN_neg/ixidn_neg_pinned_edges.parquet.gz")
veto = pd.read_parquet(D / "02S_build_IxIDN_neg/ixidn_neg_veto_edges.parquet.gz")
print("pinned negatives", len(pin), pin.d_sp.value_counts().sort_index().to_dict())
print("after veto", len(veto), veto.d_sp.value_counts().sort_index().to_dict())
ev = pd.read_parquet(RAW / "frozen/ixidn_eval_pairs.parquet")
print("evaluation pairs", len(ev), ev["class"].value_counts().to_dict())
assert pin.d_sp.min() >= 3 and pairs(veto.disease_1, veto.disease_2) <= pairs(pin.disease_1, pin.disease_2)

pinned negatives 413856 {3: 377119, 4: 35394, 5: 1323, 6: 20}
after veto 335557 {3: 301729, 4: 32500, 5: 1308, 6: 20}
evaluation pairs 378669 {'neg': 335557, 'pos': 43112}


## Matching universe and exact matching
Methods: *each positive is matched exactly to a negative on the number of trials of each of the two diseases, shared organ system, shared Orphanet clinical area, name similarity, MONDO similarity, and gene coverage, over a 985-disease evaluation universe*. The frozen universe table carries these covariates; `cellA` in the benchmark is the resulting matching cell.

In [4]:
u = pd.read_csv(RAW / "frozen/ixidn_eval_pair_universe_frozen.tsv.gz", sep="\t")
sp = pd.read_csv(RAW / "frozen/disease_split.tsv.gz", sep="\t")
print("universe pairs", len(u), u.label.value_counts().to_dict(), "| diseases", len(set(u.a) | set(u.b)), len(sp))
assert len(sp) == 985

universe pairs 224469 {0: 197054, 1: 27415} | diseases 985 985


## LLM curation and the final benchmark
The 4,302 matched candidate positives were judged and the 2,182 judged strong or plausible kept (notebook 01). Methods: *2,182 positive and 2,182 negative pairs, 4,364 pairs over 938 diseases, 179 of them common (19.1%)*.

In [5]:
b = pd.read_csv(D / "06S_final_benchmark/ixidn_matched_4364.tsv", sep="\t")
v = pd.read_parquet(RAW / "llm_annotations/ixidn_judge_verdicts.parquet")
kept = pairs(v[v.keep].disease_a, v[v.keep].disease_b)
P = b[b.label == 1]
assert pairs(P.a, P.b) == kept
dis = set(b.a) | set(b.b)
lab = pd.read_parquet(RAW / "llm_annotations/disease_labels.parquet").set_index("disease_id")
n_common = int((lab.disease_class.reindex(list(dis)) == "common").sum())
print(len(b), b.label.value_counts().to_dict(), "| diseases", len(dis), "| common", n_common, f"({n_common / len(dis):.1%})")
cells = b.groupby(["cellA", "label"]).size().unstack(fill_value=0)
print("matching cells", len(cells), "| identical pos/neg counts in every cell:", bool((cells[0] == cells[1]).all()))
assert (len(b), int(b.label.sum()), len(dis), n_common) == (4364, 2182, 938, 179)

4364 {0: 2182, 1: 2182} | diseases 938 | common 179 (19.1%)
matching cells 878 | identical pos/neg counts in every cell: True


## Graph distance of benchmark pairs
Supplementary: *every matched positive is one step apart; matched negatives are three (2,100), four (80) or five (2) steps apart*.

In [6]:
d_sp = pd.Series([nx.shortest_path_length(G, x, y) for x, y in zip(b.a, b.b)])
tab = d_sp.groupby(b.label.values).value_counts().unstack(fill_value=0)
print(tab)
assert tab.loc[1].to_dict() == {1: 2182, 3: 0, 4: 0, 5: 0} and tab.loc[0, [3, 4, 5]].tolist() == [2100, 80, 2]

      1     3   4  5
0     0  2100  80  2
1  2182     0   0  0


## Held-out test split
Methods: each disease is assigned to the test half by the parity of the MD5 hash of its identifier; a pair is in the test split when both diseases are (1,094 pairs, 491 positive / 603 negative; 477 of the 938 diseases).

In [7]:
rule = sp.disease_id.map(lambda s: "test" if int(hashlib.md5(s.encode()).hexdigest(), 16) % 2 else "tune")
print("MD5-parity rule reproduces shipped split:", bool((rule == sp.split).all()))
test = set(sp.disease_id[sp.split == "test"])
in_test = b.a.isin(test) & b.b.isin(test)
print("test pairs", int(in_test.sum()), b[in_test].label.value_counts().to_dict(), "| test diseases in benchmark", len(dis & test))
print("pools:", b.pool.value_counts().to_dict())
assert (rule == sp.split).all() and int(in_test.sum()) == 1094 and len(dis & test) == 477

MD5-parity rule reproduces shipped split: True
test pairs 1094 {0: 603, 1: 491} | test diseases in benchmark 477
pools: {'cross': 2167, 'tune': 1103, 'test': 1094}
